# Практика 2 — От LLM-приложения к агентным системам

### Цели практики


1. Построить **Reactive Agent** — агент с агентным контуром, реагирующий на текущее наблюдение
2. Построить **Planning Agent** — агент, который сначала планирует, потом действует
3. Добавить **Human-in-the-Loop** — человека в контур принятия решений

---
## 0. Подготовка

In [3]:
import warnings


import sys
import json
import math
from pathlib import Path


from dataclasses import dataclass, field
from typing import Any


from pydantic import BaseModel, Field

warnings.filterwarnings("ignore")


def _find_project_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / "pyproject.toml").exists():
            return p
    return start


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


from src.config import settings
from openai import OpenAI

In [4]:
# Polza.ai клиент (deepseek и другие модели)
polza_ai_client = OpenAI(
    base_url="https://api.polza.ai/api/v1",
    api_key=settings.polza_ai_api_key,
)

# ============================================================
# Основной клиент и модель для этой практики
# Можете заменить на любой провайдер с поддержкой tools
# ============================================================
client = polza_ai_client
# MODEL = "deepseek/deepseek-chat-v3.1"
MODEL = "openai/gpt-4.1"

# Модель для structured output (планирование)
# deepseek плохо генерирует structured output → используем GPT-4.1
PLANNER_MODEL = "openai/gpt-4.1"

---
## 1. Напоминание: LLM + Function Calling = цепочка

На прошлой практике мы научились:
- описывать доступные инструменты (tools / functions)
- получать от модели `tool_calls` — структурированный запрос на вызов
- выполнять функцию и возвращать результат
- получать финальный ответ

Это **LLM-цепочка**:

```
вопрос → [tool call → результат] → финальный ответ
```

Такая цепочка отлично работает для задач с **предсказуемым, линейным потоком**.  
Один запрос — одно (или несколько параллельных) действий — один ответ.

Но что если задача требует **неизвестного числа шагов**?

---
## 2. Подготовка

Описываем данные, тулы и тд

In [8]:
CITY_DATABASE = {
    "Москва": {
        "population": 12_600_000,
        "area_km2": 2561,
        "country": "Россия",
        "founded": 1147,
    },
    "Санкт-Петербург": {
        "population": 5_600_000,
        "area_km2": 1439,
        "country": "Россия",
        "founded": 1703,
    },
    "Новосибирск": {
        "population": 1_630_000,
        "area_km2": 505,
        "country": "Россия",
        "founded": 1893,
    },
    "Екатеринбург": {
        "population": 1_540_000,
        "area_km2": 468,
        "country": "Россия",
        "founded": 1723,
    },
    "Казань": {
        "population": 1_310_000,
        "area_km2": 614,
        "country": "Россия",
        "founded": 1005,
    },
    "Токио": {
        "population": 13_960_000,
        "area_km2": 2194,
        "country": "Япония",
        "founded": 1457,
    },
    "Пекин": {
        "population": 21_540_000,
        "area_km2": 16411,
        "country": "Китай",
        "founded": 1045,
    },
    "Нью-Йорк": {
        "population": 8_336_000,
        "area_km2": 783,
        "country": "США",
        "founded": 1624,
    },
    "Лондон": {
        "population": 8_982_000,
        "area_km2": 1572,
        "country": "Великобритания",
        "founded": 43,
    },
    "Париж": {
        "population": 2_161_000,
        "area_km2": 105,
        "country": "Франция",
        "founded": -250,
    },
}

In [9]:
TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "search_city",
            "description": "Получить информацию о городе: население (population), площадь (area_km2), страну (country), год основания (founded)",
            "parameters": {
                "type": "object",
                "properties": {
                    "city_name": {
                        "type": "string",
                        "description": "Название города на русском языке",
                    }
                },
                "required": ["city_name"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "list_cities",
            "description": "Получить список доступных городов в базе. Можно отфильтровать по названию страны",
            "parameters": {
                "type": "object",
                "properties": {
                    "country": {
                        "type": "string",
                        "description": "Название страны для фильтрации (необязательно)",
                    }
                },
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Вычислить математическое выражение. Поддерживает: +, -, *, /, **, sqrt(), round(), max(), min(), pi",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Математическое выражение, например '12600000 / 2561' или 'round(3.14159, 2)'",
                    }
                },
                "required": ["expression"],
            },
        },
    },
]

In [10]:
def search_city(city_name: str) -> dict:
    info = CITY_DATABASE.get(city_name)
    if info:
        return {"status": "found", "city": city_name, **info}
    return {
        "status": "not_found",
        "city": city_name,
        "error": f"Город '{city_name}' не найден в базе",
    }


def list_cities(country: str | None = None) -> dict:
    if country:
        cities = [name for name, data in CITY_DATABASE.items() if data["country"] == country]
    else:
        cities = list(CITY_DATABASE.keys())
    return {"cities": cities, "count": len(cities)}


def calculate(expression: str) -> dict:
    namespace = {
        "__builtins__": {},
        "math": math,
        "pi": math.pi,
        "e": math.e,
        "sqrt": math.sqrt,
        "abs": abs,
        "round": round,
        "max": max,
        "min": min,
    }
    try:
        result = eval(expression, namespace)
        return {"expression": expression, "result": result}
    except Exception as ex:
        return {"expression": expression, "error": str(ex)}


TOOL_FUNCTIONS = {
    "search_city": search_city,
    "list_cities": list_cities,
    "calculate": calculate,
}

print("Инструменты готовы:", list(TOOL_FUNCTIONS.keys()))

Инструменты готовы: ['search_city', 'list_cities', 'calculate']


In [11]:
@dataclass
class AgentStep:
    step_number: int
    tool_calls: list[dict] = field(default_factory=list)
    observations: list[dict] = field(default_factory=list)
    final_answer: str | None = None


@dataclass
class AgentTrace:
    question: str
    steps: list[AgentStep] = field(default_factory=list)
    total_steps: int = 0
    final_answer: str = ""

    def summary(self) -> str:
        lines = [f"Вопрос: {self.question}", f"Шагов: {self.total_steps}", ""]
        for step in self.steps:
            lines.append(f"--- Шаг {step.step_number} ---")
            for tool_call in step.tool_calls:
                lines.append(f"  Вызов: {tool_call['name']}({tool_call['args']})")
            for observation in step.observations:
                lines.append(f"  Результат: {observation}")
            if step.final_answer:
                lines.append(f"  Ответ: {step.final_answer}")
        lines.append(f"\nФинальный ответ: {self.final_answer}")
        return "\n".join(lines)


class PlanStep(BaseModel):
    step_number: int = Field(description="Порядковый номер шага")
    description: str = Field(description="Что нужно сделать на этом шаге")
    tool: str = Field(
        description="Какой инструмент использовать: search_city, list_cities, calculate. Если инструмент не нужен — 'none'"
    )
    tool_args: str = Field(
        default="",
        description='Аргументы для инструмента в формате JSON-строки, например: \'{"city_name": "Москва"}\'. Пустая строка "" если аргументы зависят от предыдущих шагов или инструмент не нужен.',
    )

    def get_tool_args(self) -> dict:
        """Парсит tool_args из JSON-строки в dict."""
        if not self.tool_args:
            return {}
        try:
            return json.loads(self.tool_args)
        except (json.JSONDecodeError, TypeError):
            return {}


class Plan(BaseModel):
    goal: str = Field(description="Цель — что нужно выяснить")
    reasoning: str = Field(description="Почему именно такой план")
    steps: list[PlanStep] = Field(description="Последовательность шагов для достижения цели")

---
## 3. Reactive Agent — агентный контур

### Ключевая идея

Вместо хардкода шагов мы строим **цикл**, в котором модель **сама решает**:
- какой инструмент вызвать
- с какими аргументами
- **когда остановиться** и дать финальный ответ

```
stateₜ
  ↓
decision (LLM)         ← модель решает: tool call или ответ?
  ↓
action (tool)           ← если tool call — выполняем
  ↓
observation              ← результат инструмента
  ↓
stateₜ₊₁                ← обновлённое состояние
  ↓
... (повторяем)
```

### Что такое Reactive Agent?

**Reactive Agent** — самый простой тип агента:
- решает **один шаг за раз**
- нет явного плана и долгосрочной цели
- реагирует на **текущее наблюдение** (observation)
- на каждом шаге модель смотрит на всю историю и решает: вызвать инструмент или дать ответ

### Механизм остановки

Когда модель считает, что собрала достаточно информации, она **не вызывает инструменты**, а просто отвечает текстом. Это и есть сигнал: «я закончил».

Технически: если `msg.tool_calls` пуст (или `None`) — агент завершил работу.

In [12]:
# ============================================================
# Reactive Agent — агентный контур
# ============================================================

REACTIVE_SYSTEM_PROMPT = """Ты аналитик данных. У тебя есть доступ к базе данных городов мира.

Правила:
- Всегда используй инструменты для получения данных — не придумывай числа
- Если нужно вычислить что-то — используй calculate
- Когда собрал достаточно данных — дай чёткий, структурированный ответ
- Если данных нет — честно скажи об этом"""


def run_reactive_agent(
    question: str,
    tools_schema: list | None = None,
    tool_functions: dict | None = None,
    system_prompt: str | None = None,
    max_steps: int = 15,
    verbose: bool = True,
) -> AgentTrace:
    tools_schema = tools_schema or TOOLS_SCHEMA
    tool_functions = tool_functions or TOOL_FUNCTIONS
    system_prompt = system_prompt or REACTIVE_SYSTEM_PROMPT

    trace = AgentTrace(question=question)
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]

    if verbose:
        print(f"[State] Вопрос: {question}")

    for step_num in range(1, max_steps + 1):
        if verbose:
            print(f"\n{'=' * 50}")
            print(f"--- Шаг {step_num} ---")

        # Decision: LLM решает — вызвать инструмент или дать ответ
        if verbose:
            print(f"  [Decision] LLM анализирует состояние...")

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools_schema,
            temperature=0,
        )
        msg = response.choices[0].message
        messages.append(msg)

        step = AgentStep(step_number=step_num)

        # Нет вызовов инструментов — модель дала финальный ответ
        if not msg.tool_calls:
            step.final_answer = msg.content
            trace.steps.append(step)
            trace.final_answer = msg.content or ""
            trace.total_steps = step_num
            if verbose:
                print(f"  [Decision] → Финальный ответ (нет tool_calls)")
                print(f"  [Result] {msg.content}")
            break

        # Выполняем все вызовы инструментов
        if verbose:
            print(f"  [Decision] → Вызвать {len(msg.tool_calls)} инструмент(ов)")

        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)

            if verbose:
                print(f"  [Action] {name}({args})")

            result = tool_functions[name](**args)

            if verbose:
                print(f"  [Observation] {result}")

            step.tool_calls.append({"name": name, "args": args})
            step.observations.append(result)

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result, ensure_ascii=False),
                }
            )

        if verbose:
            print(f"  [State_i] Состояние обновлено (messages: {len(messages)})")

        trace.steps.append(step)
    else:
        trace.total_steps = max_steps
        trace.final_answer = "Достигнут лимит шагов"

    return trace

### 3.1 Запускаем агент на простом вопросе

Начнём с простого: один tool call — один ответ.

In [13]:
# Простой вопрос: один tool call — один ответ
trace = run_reactive_agent("Какое население Токио?")

print(f"\nИтого шагов: {trace.total_steps}")

[State] Вопрос: Какое население Токио?

--- Шаг 1 ---
  [Decision] LLM анализирует состояние...
  [Decision] → Вызвать 1 инструмент(ов)
  [Action] search_city({'city_name': 'Токио'})
  [Observation] {'status': 'found', 'city': 'Токио', 'population': 13960000, 'area_km2': 2194, 'country': 'Япония', 'founded': 1457}
  [State_i] Состояние обновлено (messages: 4)

--- Шаг 2 ---
  [Decision] LLM анализирует состояние...
  [Decision] → Финальный ответ (нет tool_calls)
  [Result] Население Токио составляет 13 960 000 человек.

Итого шагов: 2


<img src="pictures/Reactive.JPG" width="400"/>

### 3.2 Запускаем агент на сложном вопросе

Теперь та самая задача, которая ломала цепочку: **неизвестное число шагов**.

In [14]:
# Сложный вопрос: неизвестное число шагов, динамический поиск
trace = run_reactive_agent(
    "Какой город России имеет самую высокую плотность населения? Плотность = население / площадь."
)

print("\n" + "=" * 60)
print("ТРАССИРОВКА")
print("=" * 60)
print(trace.summary())

[State] Вопрос: Какой город России имеет самую высокую плотность населения? Плотность = население / площадь.

--- Шаг 1 ---
  [Decision] LLM анализирует состояние...
  [Decision] → Вызвать 1 инструмент(ов)
  [Action] list_cities({'country': 'Россия'})
  [Observation] {'cities': ['Москва', 'Санкт-Петербург', 'Новосибирск', 'Екатеринбург', 'Казань'], 'count': 5}
  [State_i] Состояние обновлено (messages: 4)

--- Шаг 2 ---
  [Decision] LLM анализирует состояние...
  [Decision] → Вызвать 5 инструмент(ов)
  [Action] search_city({'city_name': 'Москва'})
  [Observation] {'status': 'found', 'city': 'Москва', 'population': 12600000, 'area_km2': 2561, 'country': 'Россия', 'founded': 1147}
  [Action] search_city({'city_name': 'Санкт-Петербург'})
  [Observation] {'status': 'found', 'city': 'Санкт-Петербург', 'population': 5600000, 'area_km2': 1439, 'country': 'Россия', 'founded': 1703}
  [Action] search_city({'city_name': 'Новосибирск'})
  [Observation] {'status': 'found', 'city': 'Новосибирск', '

### Что здесь произошло?

Обратите внимание:

1. **Мы не писали цикл по городам** — модель сама решила, что нужно получить данные для каждого города
2. **Мы не писали логику сравнения** — модель сама сравнила результаты
3. **Мы не указывали число шагов** — модель сама определила, когда данных достаточно
4. **Код `run_reactive_agent` — универсальный** — он работает для любого вопроса

Это и есть **агентный контур**: модель управляет процессом, а не код.

In [15]:
# Ещё одна задача — совсем другого типа, но с тем же кодом!
trace3 = run_reactive_agent("Сравни Лондон и Париж: какой город старше и во сколько раз больше по площади?")

print("\n" + "=" * 60)
print("ТРАССИРОВКА")
print("=" * 60)
print(trace3.summary())

[State] Вопрос: Сравни Лондон и Париж: какой город старше и во сколько раз больше по площади?

--- Шаг 1 ---
  [Decision] LLM анализирует состояние...
  [Decision] → Вызвать 2 инструмент(ов)
  [Action] search_city({'city_name': 'Лондон'})
  [Observation] {'status': 'found', 'city': 'Лондон', 'population': 8982000, 'area_km2': 1572, 'country': 'Великобритания', 'founded': 43}
  [Action] search_city({'city_name': 'Париж'})
  [Observation] {'status': 'found', 'city': 'Париж', 'population': 2161000, 'area_km2': 105, 'country': 'Франция', 'founded': -250}
  [State_i] Состояние обновлено (messages: 5)

--- Шаг 2 ---
  [Decision] LLM анализирует состояние...
  [Decision] → Вызвать 1 инструмент(ов)
  [Action] calculate({'expression': '1572 / 105'})
  [Observation] {'expression': '1572 / 105', 'result': 14.971428571428572}
  [State_i] Состояние обновлено (messages: 7)

--- Шаг 3 ---
  [Decision] LLM анализирует состояние...
  [Decision] → Финальный ответ (нет tool_calls)
  [Result] - Париж стар

### Характеристики Reactive Agent

| Свойство | Reactive Agent |
|----------|---------------|
| План | Нет |
| Стратегия | Реагирует на текущее наблюдение |
| Возврат назад | Нет (идёт только вперёд) |
| Сложность | Низкая |
| Отладка | Простая (смотрим трассировку) |
| Хороший baseline | Да |

**Когда использовать?**
- Задачи с понятными инструментами
- Не требуется долгосрочное планирование
- Нужно быстро и просто

**Когда не подходит?**
- Сложные задачи с зависимостями между шагами
- Задачи, где важна стратегия
- Когда нужна оптимизация порядка действий

In [19]:
trace3 = run_reactive_agent(
    """Какой процент от суммарного населения всех городов в базе приходится на города, 
    основанные до нашей эры?"""
)

print("\n" + "=" * 60)
print("ТРАССИРОВКА")
print("=" * 60)
print(trace3.summary())

[State] Вопрос: Какой процент от суммарного населения всех городов в базе приходится на города, 
    основанные до нашей эры?

--- Шаг 1 ---
  [Decision] LLM анализирует состояние...
  [Decision] → Вызвать 1 инструмент(ов)
  [Action] list_cities({'country': ''})
  [Observation] {'cities': ['Москва', 'Санкт-Петербург', 'Новосибирск', 'Екатеринбург', 'Казань', 'Токио', 'Пекин', 'Нью-Йорк', 'Лондон', 'Париж'], 'count': 10}
  [State_i] Состояние обновлено (messages: 4)

--- Шаг 2 ---
  [Decision] LLM анализирует состояние...
  [Decision] → Вызвать 10 инструмент(ов)
  [Action] search_city({'city_name': 'Москва'})
  [Observation] {'status': 'found', 'city': 'Москва', 'population': 12600000, 'area_km2': 2561, 'country': 'Россия', 'founded': 1147}
  [Action] search_city({'city_name': 'Санкт-Петербург'})
  [Observation] {'status': 'found', 'city': 'Санкт-Петербург', 'population': 5600000, 'area_km2': 1439, 'country': 'Россия', 'founded': 1703}
  [Action] search_city({'city_name': 'Новосибирск'}

---
## 4. Planning Agent

### Идея

**Planning Agent** разделяет процесс на фазы:

1. **Планирование (Planner)** — LLM анализирует задачу и создаёт план действий
2. **Исполнение** — код выполняет план шаг за шагом: Action → Observation → State_i
3. **Оценка (Evaluation)** — LLM оценивает, достигнута ли цель
4. **Replan** — если цель не достигнута, Planner пересматривает план

```
Goal → State → Planner (LLM) → Plan → [Action → Observation → State_i] → Evaluation (LLM) → Result
                  ↑                                                             |
                  └──────────────── Replan (если цель не достигнута) ───────────┘
```

В отличие от Reactive Agent, здесь модель **сначала думает целиком**, а потом действует по плану.
В отличие от Planner–Executor, здесь **нет отдельного LLM-исполнителя** — шаги плана исполняются напрямую кодом.

### Зачем?

- Модель может оценить задачу **целиком** перед началом
- План можно **проверить** (людьми или кодом) до выполнения
- Если шаг провалился — план можно **пересмотреть** (Replan)
- Более предсказуемое и контролируемое поведение

In [16]:
# ============================================================
# Planning Agent — полный флоу в одной функции
# Goal → State → Planner (LLM) → Plan → [Action → Observation → State_i] → Evaluation (LLM) → Result / Replan
# ============================================================

DEFAULT_PLANNER_SYSTEM_PROMPT = """Ты планировщик задач. Тебе дана задача и список доступных инструментов.

Доступные инструменты:
1. search_city(city_name: str) — информация о городе (население, площадь, страна, год основания)
2. list_cities(country: str | None) — список городов, опционально по стране
3. calculate(expression: str) — вычислить математическое выражение

Составь пошаговый план решения задачи. Каждый шаг должен содержать:
- описание действия
- какой инструмент использовать (или 'none')
- аргументы для инструмента в виде JSON-строки (например: '{"city_name": "Москва"}')

Если на каком-то шаге аргументы зависят от результата предыдущего шага,
укажи это в описании, а в tool_args оставь пустую строку "".

Важно: план должен быть конкретным и исполняемым."""

DEFAULT_EVALUATION_PROMPT = """Ты оцениваешь результаты выполнения плана.

Вопрос пользователя: {question}

Цель плана: {goal}

Исходный план:
{plan_text}

Результаты выполнения:
{results_text}

Оцени:
1. Все ли шаги выполнены успешно?
2. Достаточно ли собранных данных для ответа на вопрос?
3. Нужно ли пересмотреть план?

Ответь в формате JSON:
{{"goal_achieved": true/false, "reasoning": "...", "final_answer": "..." или null}}

Если goal_achieved=true, дай развёрнутый final_answer:
- Укажи конкретный ответ на вопрос
- Приведи ключевые числа и данные из результатов
- Покажи промежуточные вычисления, если они были
- Сделай краткий вывод

Если goal_achieved=false, объясни что не хватает в reasoning."""


def _resolve_via_llm(client, model, step, all_results, tools_schema, tool_functions, verbose):
    """Просит LLM подставить правильные аргументы для шага, зависящего от контекста."""
    results_text = json.dumps(all_results, ensure_ascii=False, indent=2)
    resolve_prompt = (
        f"Текущий шаг плана:\n"
        f"  Описание: {step.description}\n"
        f"  Инструмент: {step.tool}\n\n"
        f"Результаты предыдущих шагов:\n{results_text}\n\n"
        f"Вызови нужный инструмент с правильными аргументами, "
        f"подставив данные из предыдущих результатов."
    )
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "Вызови нужный инструмент с правильными аргументами на основе контекста.",
            },
            {"role": "user", "content": resolve_prompt},
        ],
        tools=tools_schema,
        temperature=0,
    )
    msg = resp.choices[0].message

    if msg.tool_calls:
        step_results = []
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)
            if verbose:
                print(f"  [Action] {name}({args})")
            func = tool_functions.get(name)
            if func:
                result = func(**args)
            else:
                result = {"error": f"Неизвестный инструмент: {name}"}
            if verbose:
                print(f"  [Observation] {json.dumps(result, ensure_ascii=False)}")
            step_results.append(result)
        return step_results[0] if len(step_results) == 1 else step_results
    else:
        if verbose:
            print(f"  [Skip] LLM не вызвал инструмент")
        return "skipped (LLM did not call tool)"


def run_planning_agent(
    question: str,
    client=client,
    model=MODEL,
    planner_model=PLANNER_MODEL,
    tool_functions=TOOL_FUNCTIONS,
    tools_schema=TOOLS_SCHEMA,
    planner_system_prompt=DEFAULT_PLANNER_SYSTEM_PROMPT,
    evaluation_prompt=DEFAULT_EVALUATION_PROMPT,
    max_replans=2,
    verbose=True,
) -> dict:
    all_results = []
    replan_count = 0

    # ── Фаза 1: Planner (LLM) генерирует план ──
    if verbose:
        print(f"[Goal] Вопрос: {question}")
        print(f"\n{'=' * 60}")
        print(f"[Planner] LLM ({planner_model}) генерирует план...")

    response = client.chat.completions.parse(
        model=planner_model,
        messages=[
            {"role": "system", "content": planner_system_prompt},
            {"role": "user", "content": question},
        ],
        response_format=Plan,
        temperature=0,
    )
    current_plan = response.choices[0].message.parsed

    if verbose:
        print(f"  Цель: {current_plan.goal}")
        print(f"  Обоснование: {current_plan.reasoning}")
        print(f"  Шаги ({len(current_plan.steps)}):")
        for step in current_plan.steps:
            tool_args = step.get_tool_args()
            args_str = tool_args if tool_args else "{зависит от предыдущих шагов}"
            print(f"    {step.step_number}. {step.description}")
            print(f"       Инструмент: {step.tool}({args_str})")

    # ── Фаза 2: Исполнение + Evaluation + Replan ──
    while True:
        if verbose:
            print(f"\n{'=' * 60}")
            print(f"[State] Текущий план: {current_plan.goal}")
            print(f"  Шагов в плане: {len(current_plan.steps)}")
            print(f"  Результатов собрано: {len(all_results)}")
            print("=" * 60)

        # ── Исполнение шагов плана: [Action → Observation → State_i] ──
        for step in current_plan.steps:
            step_num = step.step_number
            tool_args = step.get_tool_args()

            if verbose:
                print(f"\n{'─' * 50}")
                print(f"--- Шаг {step_num}: {step.description} ---")

            if step.tool == "none":
                if verbose:
                    print(f"  [Skip] Шаг без инструмента")
                all_results.append(
                    {
                        "step": step_num,
                        "description": step.description,
                        "result": "skipped (no tool needed)",
                    }
                )
                continue

            # Если аргументы заданы в плане — пробуем вызвать напрямую
            if tool_args:
                tool_name = step.tool
                if verbose:
                    print(f"  [Action] {tool_name}({tool_args})")
                func = tool_functions.get(tool_name)
                if func:
                    result = func(**tool_args)
                else:
                    result = {"error": f"Неизвестный инструмент: {tool_name}"}

                # Если вызов вернул ошибку — fallback на LLM-резолв
                if isinstance(result, dict) and "error" in result:
                    if verbose:
                        print(f"  [Error] {result['error']}")
                        print(f"  [Resolve] Ошибка при прямом вызове — LLM разрешает аргументы...")
                    result = _resolve_via_llm(
                        client,
                        model,
                        step,
                        all_results,
                        tools_schema,
                        tool_functions,
                        verbose,
                    )
                else:
                    if verbose:
                        print(f"  [Observation] {json.dumps(result, ensure_ascii=False)}")

                all_results.append(
                    {
                        "step": step_num,
                        "description": step.description,
                        "result": result,
                    }
                )
            else:
                # Аргументы зависят от предыдущих шагов — просим LLM подставить
                if verbose:
                    print(f"  [Resolve] Аргументы зависят от предыдущих шагов — LLM разрешает...")
                result = _resolve_via_llm(
                    client,
                    model,
                    step,
                    all_results,
                    tools_schema,
                    tool_functions,
                    verbose,
                )
                all_results.append(
                    {
                        "step": step_num,
                        "description": step.description,
                        "result": result,
                    }
                )

            if verbose:
                print(f"  [State_i] Состояние обновлено (результатов: {len(all_results)})")

        # ── Evaluation (LLM): оценка — цель достигнута? ──
        if verbose:
            print(f"\n{'=' * 60}")
            print(f"[Evaluation] LLM оценивает результаты...")

        results_text = json.dumps(all_results, ensure_ascii=False, indent=2)
        plan_text = current_plan.model_dump_json(indent=2)

        eval_response = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "system",
                    "content": "Ты оцениваешь результаты выполнения плана. Отвечай строго в формате JSON.",
                },
                {
                    "role": "user",
                    "content": evaluation_prompt.format(
                        question=question,
                        goal=current_plan.goal,
                        plan_text=plan_text,
                        results_text=results_text,
                    ),
                },
            ],
            temperature=0,
        )

        eval_text = eval_response.choices[0].message.content
        eval_text_clean = eval_text.strip()
        if eval_text_clean.startswith("```"):
            eval_text_clean = eval_text_clean.split("\n", 1)[1].rsplit("```", 1)[0].strip()

        try:
            evaluation = json.loads(eval_text_clean)
        except json.JSONDecodeError:
            evaluation = {
                "goal_achieved": True,
                "reasoning": "Не удалось распарсить оценку",
                "final_answer": eval_text,
            }

        if verbose:
            print(f"  [Evaluation] Цель достигнута: {evaluation.get('goal_achieved')}")
            print(f"  [Evaluation] Обоснование: {evaluation.get('reasoning')}")

        # ── Result: если цель достигнута — Goal Achieved ──
        if evaluation.get("goal_achieved"):
            answer = evaluation.get("final_answer", "")
            if verbose:
                print(f"\n{'=' * 60}")
                print(f"[Result] Goal Achieved!")
                print(f"\nФинальный ответ: {answer}")
            return {"plan": current_plan, "results": all_results, "answer": answer}

        # ── Replan: возврат к Planner ──
        replan_count += 1
        if replan_count > max_replans:
            if verbose:
                print(f"\n[Replan] Лимит пересмотров плана исчерпан ({max_replans})")
            final_response = client.chat.completions.create(
                model=model,
                messages=[
                    {
                        "role": "system",
                        "content": "Дай лучший возможный ответ на основе собранных данных.",
                    },
                    {
                        "role": "user",
                        "content": f"Вопрос: {question}\n\nДанные:\n{results_text}",
                    },
                ],
                temperature=0,
            )
            answer = final_response.choices[0].message.content
            if verbose:
                print(f"\nФинальный ответ (после лимита): {answer}")
            return {"plan": current_plan, "results": all_results, "answer": answer}

        if verbose:
            print(f"\n{'=' * 60}")
            print(f"[Replan] Цель не достигнута — пересматриваем план (попытка {replan_count}/{max_replans})...")

        replan_prompt = (
            f"{planner_system_prompt}\n\n"
            f"Предыдущий план не привёл к полному результату.\n"
            f"Причина: {evaluation.get('reasoning')}\n\n"
            f"Уже собранные данные:\n{results_text}\n\n"
            f"Составь НОВЫЙ план, учитывая уже собранные данные. "
            f"Не повторяй шаги, результаты которых уже есть."
        )
        response = client.chat.completions.parse(
            model=planner_model,
            messages=[
                {"role": "system", "content": replan_prompt},
                {"role": "user", "content": question},
            ],
            response_format=Plan,
            temperature=0,
        )
        current_plan = response.choices[0].message.parsed

        if verbose:
            print(f"[Planner] Новый план:")
            print(f"  Цель: {current_plan.goal}")
            for step in current_plan.steps:
                tool_args = step.get_tool_args()
                print(f"  {step.step_number}. {step.description} → {step.tool}({tool_args})")

<img src="pictures/Planner.JPG" width="400"/>

In [17]:
# Запускаем Planning Agent — полный флоу в одном вызове
result = run_planning_agent(
    "Какой город России имеет самую высокую плотность населения? Плотность = население / площадь."
)

[Goal] Вопрос: Какой город России имеет самую высокую плотность населения? Плотность = население / площадь.

[Planner] LLM (openai/gpt-4.1) генерирует план...
  Цель: Найти город России с самой высокой плотностью населения (плотность = население / площадь).
  Обоснование: Для решения задачи нужно получить список городов России, затем для каждого города узнать его население и площадь, рассчитать плотность, и выбрать город с максимальной плотностью.
  Шаги (4):
    1. Получить список всех городов России.
       Инструмент: list_cities({'country': 'Россия'})
    2. Для каждого города из списка получить данные о населении и площади.
       Инструмент: search_city({'city_name': '<название города из списка>'})
    3. Для каждого города рассчитать плотность населения (население / площадь).
       Инструмент: calculate({'expression': '<население>/<площадь>'})
    4. Сравнить плотности всех городов и выбрать город с максимальной плотностью.
       Инструмент: none({зависит от предыдущих шагов})